# Action distribution figure

Per-prompt strip plot of the share of tutor actions across the 12 action
categories A–L (M = Other is dropped). Each category column holds one
marker per LM tutor (color + shape uniquely identify the model) plus a
dashed horizontal baseline at the human level for that category.

Consumes the classified facet CSVs produced by `tutorsim taxonomy classify`.
Requires `matplotlib` beyond the `taxonomy` extras.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np

from tutorsim import taxonomy as tx

REPO = Path.cwd()
while not (REPO / ".git").exists() and REPO != REPO.parent:
    REPO = REPO.parent

# Update these to point at the classified.csv files produced by
#   tutorsim taxonomy classify --output <dir>
HUMAN_CLASSIFIED = REPO / "data" / "taxonomy" / "human" / "classified.csv"
LM_CLASSIFIED    = REPO / "data" / "taxonomy" / "lm"    / "classified.csv"
OUT_PDF          = REPO / "analysis" / "working-paper-20260630" / "figure_action_distribution.pdf"


In [ ]:
# Per-LM color + marker shape. Okabe-Ito colorblind-safe palette plus one
# pink. Markers omit circle ('o') -- reserved for the human row in the
# companion dumbbell figure.
MODEL_DISPLAY = {
    "claude-opus-4-8":             "Claude Opus 4.8",
    "claude-sonnet-4-6":           "Claude Sonnet 4.6",
    "deepseek-ai/DeepSeek-V4-Pro": "DeepSeek V4 Pro",
    "gemini-2.5-pro":              "Gemini 2.5 Pro",
    "gemini-3.5-flash":            "Gemini 3.5 Flash",
    "gpt-5.5-2026-04-23":          "GPT 5.5",
    "gpt-5.4-mini-2026-03-17":     "GPT 5.4 mini",
}
MODEL_COLORS = {
    "claude-opus-4-8":             "#D55E00",
    "claude-sonnet-4-6":           "#E69F00",
    "deepseek-ai/DeepSeek-V4-Pro": "#CC79A7",
    "gemini-2.5-pro":              "#0072B2",
    "gemini-3.5-flash":            "#56B4E9",
    "gpt-5.5-2026-04-23":          "#009E73",
    "gpt-5.4-mini-2026-03-17":     "#F0529C",
}
MODEL_MARKERS = {
    "claude-opus-4-8":             "v",
    "claude-sonnet-4-6":           "s",
    "deepseek-ai/DeepSeek-V4-Pro": "^",
    "gemini-2.5-pro":              "D",
    "gemini-3.5-flash":            "p",
    "gpt-5.5-2026-04-23":          "P",
    "gpt-5.4-mini-2026-03-17":     "X",
}

# Short axis labels per category letter. M (Other) is dropped from the plot.
SHORT_LABELS = {
    "A": "Guiding\nquestions",
    "B": "Breaking\ndown",
    "C": "Explaining",
    "D": "Alt. reps.",
    "E": "Hints",
    "F": "Supplying\nanswers",
    "G": "Prompting\njustification",
    "H": "Independent\nwork",
    "I": "Increasing\ncomplexity",
    "J": "Self-\nassessment",
    "K": "Affirmations",
    "L": "Transitioning",
}
PLOTTED_LETTERS = list(SHORT_LABELS)  # all letters except M
BASELINE_COLOR  = "#1C2B33"


In [ ]:
human = tx.read_classified_csv(HUMAN_CLASSIFIED)
lm    = tx.read_classified_csv(LM_CLASSIFIED)

human_df = tx.facets_to_dataframe(human)
lm_df    = tx.facets_to_dataframe(lm)

# Build the human-rate series, then resort the plotted letters in
# DECREASING human-baseline order. The downstream code treats
# PLOTTED_LETTERS as the canonical x-axis order, so reassigning it here
# is enough to flip the whole figure.
_initial_letters = list(SHORT_LABELS)
human_pct = (
    tx.macro_distribution(human_df)
      .set_index("letter")["mean_pct"]
      .reindex(_initial_letters, fill_value=0.0)
      .sort_values(ascending=False)
)
PLOTTED_LETTERS = list(human_pct.index)

lm_dist = tx.macro_distribution(lm_df, group_keys=("model", "prompt"))
print(f"human categories sorted by rate (desc): {PLOTTED_LETTERS}")
print(f"lm cells: {lm_dist.groupby(['model','prompt']).ngroups}")


In [ ]:
def plot_strip(ax, prompt, x_offsets):
    sub = lm_dist[lm_dist["prompt"] == prompt]
    n_cat = len(PLOTTED_LETTERS)
    x = np.arange(n_cat)
    for xi in x[:-1]:
        ax.axvline(xi + 0.5, color=BASELINE_COLOR, alpha=0.10, lw=0.8, zorder=0)
    for xi, letter in zip(x, PLOTTED_LETTERS):
        h = float(human_pct[letter])
        ax.hlines(h, xi - 0.36, xi + 0.36, color=BASELINE_COLOR, ls="--", lw=1.4, zorder=3)
    for off, model_id in zip(x_offsets, MODEL_DISPLAY):
        vals = []
        for letter in PLOTTED_LETTERS:
            row = sub[(sub["model"] == model_id) & (sub["letter"] == letter)]
            vals.append(float(row["mean_pct"].iloc[0]) if len(row) else 0.0)
        ax.scatter(x + off, vals, s=80,
                   color=MODEL_COLORS[model_id], marker=MODEL_MARKERS[model_id],
                   edgecolor="white", linewidth=0.6, zorder=4)
    ax.set_xticks(x)
    ax.set_xticklabels([SHORT_LABELS[L] for L in PLOTTED_LETTERS],
                       rotation=0, ha="center", fontsize=9)
    ax.set_xlim(-0.5, n_cat - 0.5)
    ax.set_ylabel("Share of tutor actions (%)", fontsize=11)
    ax.tick_params(axis="y", labelsize=10)
    ax.grid(axis="y", ls=":", alpha=0.4)
    ax.set_axisbelow(True)


In [ ]:
PROMPTS = [("plain", "Plain prompt"),
           ("scaffolding_rigor", "Evaluation-aware prompt")]
x_offsets = np.linspace(-0.30, 0.30, len(MODEL_DISPLAY))

# Make the figure a touch taller and use plt.subplots_adjust instead of
# constrained_layout so we can carve out explicit bottom margin for the
# legend without it clipping the x-tick labels.
fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(13, 8.5),
                         sharex=True, sharey=True)

ymax = max(float(lm_dist["mean_pct"].max()), float(human_pct.max()))
for ax, (prompt, label) in zip(axes, PROMPTS):
    plot_strip(ax, prompt, x_offsets)
    ax.set_title(label, fontsize=12, loc="left")
    ax.set_ylim(0, ymax * 1.08)

handles = [Line2D([0], [0], color=BASELINE_COLOR, ls="--", lw=1.4,
                  label="Human (per-category baseline)")]
for model_id, display in MODEL_DISPLAY.items():
    handles.append(Line2D([0], [0], marker=MODEL_MARKERS[model_id],
                          linestyle="none",
                          markerfacecolor=MODEL_COLORS[model_id],
                          markeredgecolor="white", markersize=10,
                          label=display))

# Reserve room at the bottom for the x-tick labels AND the legend.
fig.subplots_adjust(left=0.07, right=0.98, top=0.95, bottom=0.18, hspace=0.30)
fig.legend(handles=handles, loc="lower center",
           ncol=4, fontsize=10, frameon=False,
           bbox_to_anchor=(0.5, 0.02))

fig.savefig(OUT_PDF, dpi=150, bbox_inches="tight")
print(f"wrote {OUT_PDF.relative_to(REPO)}")
fig
